In [1]:
import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [5]:
def b2_evaluation_scores(baselines):
    """Return baseline rows with (template_name, score) MultiIndex columns.

    Read b2_four_baselines_v1 results from the repository outputs directory.
    For each template, average image scores within each seed, then give
    each seed equal weight. BERT uses BERTScore F1. FID uses template-level
    records only, excluding optional prompt-level records.
    """
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    if isinstance(baselines, str):
        baselines = [baselines]
    baselines = list(dict.fromkeys(baselines))
    output_root = Path("/aiau010_scratch/azm0269/clover/outputs")
    score_fields = {
        "fid": "fid",
        "bert": "bert_f1",
        "clip": "clip",
        "imagereward": "imagereward",
    }
    rows = []
    templates = []
    for baseline in baselines:
        folder = output_root / baseline / "evals/b2/b2_four_baselines_v1"
        scores = {}
        expected_groups = None
        for metric, field in score_fields.items():
            path = folder / f"{metric}.json"
            records = json.loads(path.read_text())["records"]
            if metric == "fid":
                records = [record for record in records if record["level"] == "template"]
            frame = pd.DataFrame(records)
            if frame.empty or not {"seed", "template_name", field}.issubset(frame.columns):
                raise ValueError(f"Missing template, seed, or score records in {path}")
            if frame[["seed", "template_name"]].isna().any().any() or not np.isfinite(frame[field].to_numpy(dtype=float)).all():
                raise ValueError(f"Missing template/seed or non-finite scores in {path}")
            groups = set(zip(frame["template_name"], frame["seed"]))
            if expected_groups is not None and groups != expected_groups:
                raise ValueError(f"Metric template/seed sets do not match for {baseline}: {path}")
            expected_groups = groups
            per_seed = frame.groupby(["template_name", "seed"], sort=False)[field].mean()
            for template, value in per_seed.groupby(level="template_name", sort=False).mean().items():
                if template not in templates:
                    templates.append(template)
                scores[(template, metric)] = value
        rows.append(scores)

    columns = pd.MultiIndex.from_product(
        [templates, list(score_fields)], names=["template_name", "score"]
    )
    return pd.DataFrame(
        rows, index=pd.Index(baselines, name="baseline"), columns=columns
    )


In [8]:
### Lower FID is good for diversity and quality. It more closely resembles the true distrubution.
## Methods producing higher FID can be suffering from mode collapse as their generation quality starts conforming to narrow distributions.

In [7]:
scores = b2_evaluation_scores(["ddpo", "b2diffurl", "emo_v3_best_run", "sd15"])
scores.round(2).T

baseline                               ddpo  b2diffurl  emo_v3_best_run  sd15
template_name           score                                                
object_behavior         fid          151.06     135.05            99.38  0.00
                        bert           0.67       0.63             0.63  0.62
                        clip           0.39       0.38             0.34  0.34
                        imagereward    0.57       0.57            -0.06 -0.21
object_attribute        fid          162.73      72.54            55.95  0.00
                        bert           0.49       0.50             0.47  0.47
                        clip           0.29       0.32             0.32  0.32
                        imagereward   -0.53       0.34             0.15  0.19
positional_relationship fid          197.98     152.01            97.95  0.00
                        bert           0.50       0.48             0.49  0.49
                        clip           0.26       0.26             0.26  0.26
                        imagereward   -1.39      -1.35            -1.27 -1.25